# ADK Travel Planner — Research Notebook

Multi-agent travel planner using Google ADK. Agents are defined and orchestrated
inline — no external servers required.

**Architecture:**
```
Request → Host Agent → Flight Agent
                    → Stay Agent
                    → Activities Agent
```

Set `DEEPSEEK_API_KEY` in your environment for real DeepSeek responses.
Without it, the notebook falls back to mock data.

In [10]:
import os
import json
from pprint import pprint

HAS_API_KEY = bool(os.getenv("DEEPSEEK_API_KEY"))

---
## Define agents inline

In [11]:
async def flight_agent(request):
    origin = request["origin"]
    dest = request["destination"]
    budget = request["budget"]

    if HAS_API_KEY:
        from google.adk.agents import Agent as AdkAgent
        from google.adk.models.lite_llm import LiteLlm
        from google.adk.runners import Runner
        from google.adk.sessions import InMemorySessionService
        from google.genai import types

        agent = AdkAgent(
            name="flight_agent",
            model=LiteLlm("deepseek/deepseek-chat"),
            instruction=(
                f"Given a trip from {origin} to {dest} with budget ${budget}, "
                "suggest 2-3 realistic flight options with airline, price, and times."
            )
        )
        svc = InMemorySessionService()
        runner = Runner(agent=agent, app_name="flight", session_service=svc)
        await svc.create_session(app_name="flight", user_id="u1", session_id="s1")

        prompt = f"Suggest flights from {origin} to {dest} within ${budget}."
        msg = types.Content(role="user", parts=[types.Part(text=prompt)])
        async for event in runner.run_async(user_id="u1", session_id="s1", new_message=msg):
            if event.is_final_response():
                return {"flights": event.content.parts[0].text}
        return {"flights": "No response from agent."}

    # Mock fallback
    return {
        "flights": (
            f"1. {['Delta','United','American'][hash(dest)%3]} — ${budget-500} direct 8h\n"
            f"2. {['JetBlue','Alaska','Southwest'][hash(origin)%3]} — ${budget-300} 1 stop 10h"
        )
    }

In [12]:
async def stay_agent(request):
    dest = request["destination"]
    budget = request["budget"]

    if HAS_API_KEY:
        from google.adk.agents import Agent as AdkAgent
        from google.adk.models.lite_llm import LiteLlm
        from google.adk.runners import Runner
        from google.adk.sessions import InMemorySessionService
        from google.genai import types

        agent = AdkAgent(
            name="stay_agent",
            model=LiteLlm("deepseek/deepseek-chat"),
            instruction=f"Suggest hotels in {dest} within ${budget} total."
        )
        svc = InMemorySessionService()
        runner = Runner(agent=agent, app_name="stay", session_service=svc)
        await svc.create_session(app_name="stay", user_id="u1", session_id="s1")

        msg = types.Content(role="user", parts=[types.Part(text=f"Hotels in {dest} under ${budget}?")])
        async for event in runner.run_async(user_id="u1", session_id="s1", new_message=msg):
            if event.is_final_response():
                return {"stays": event.content.parts[0].text}
        return {"stays": "No response."}

    return {
        "stays": (
            f"1. Grand {dest} Hotel — ${int(budget*0.3)}/night downtown\n"
            f"2. {dest} Budget Inn — ${int(budget*0.15)}/night near transit"
        )
    }

In [13]:
async def activities_agent(request):
    dest = request["destination"]
    budget = request["budget"]

    if HAS_API_KEY:
        from google.adk.agents import Agent as AdkAgent
        from google.adk.models.lite_llm import LiteLlm
        from google.adk.runners import Runner
        from google.adk.sessions import InMemorySessionService
        from google.genai import types

        agent = AdkAgent(
            name="activities_agent",
            model=LiteLlm("deepseek/deepseek-chat"),
            instruction=f"Suggest activities in {dest} within ${budget}."
        )
        svc = InMemorySessionService()
        runner = Runner(agent=agent, app_name="activities", session_service=svc)
        await svc.create_session(app_name="activities", user_id="u1", session_id="s1")

        msg = types.Content(role="user", parts=[types.Part(text=f"Things to do in {dest}?")])
        async for event in runner.run_async(user_id="u1", session_id="s1", new_message=msg):
            if event.is_final_response():
                return {"activities": event.content.parts[0].text}
        return {"activities": "No response."}

    return {
        "activities": (
            f"1. {dest} City Tour — ${int(budget*0.1)} (3h)\n"
            f"2. Local Food Tasting — ${int(budget*0.05)} (2h)"
        )
    }

---
## Orchestrator — calls all three agents

In [14]:
async def host_agent(request):
    flights = await flight_agent(request)
    stays = await stay_agent(request)
    activities = await activities_agent(request)
    return {
        "flights": flights.get("flights", "N/A"),
        "stay": stays.get("stays", "N/A"),
        "activities": activities.get("activities", "N/A")
    }

---
## Run a trip

In [ ]:
trip = {
    "origin": "New York",
    "destination": "Tokyo",
    "start_date": "2026-09-15",
    "end_date": "2026-09-25",
    "budget": 4000
}

result = await host_agent(trip)

print("=== FLIGHTS ===")
print(result["flights"])
print("\n=== STAYS ===")
print(result["stay"])
print("\n=== ACTIVITIES ===")
print(result["activities"])

---
## Experiment — change the trip

In [ ]:
trip2 = {
    "origin": "London",
    "destination": "Bangkok",
    "start_date": "2026-11-01",
    "end_date": "2026-11-14",
    "budget": 2500
}

result2 = await host_agent(trip2)

print("=== FLIGHTS ===")
print(result2["flights"])
print("\n=== STAYS ===")
print(result2["stay"])
print("\n=== ACTIVITIES ===")
print(result2["activities"])

---
## How it works

1. **Orchestrator** (`host_agent`) receives a request with origin, destination, dates, budget
2. Dispatches to three domain **sub-agents** — each uses ADK + LiteLLM + DeepSeek (or mock)
3. Each sub-agent returns structured suggestions for its domain
4. Orchestrator collects all responses and returns a combined result

The A2A (Agent-to-Agent) protocol in the real system does this over HTTP.
This notebook does it with direct function calls for simplicity and reproducibility.